# The "Anchor" Similarity Method (Most Lightweight)

This uses a small embedding model (like all-MiniLM-L6-v2) which can run on a standard CPU. Instead of "extracting" text, you compare the entire review to specific "anchor" concepts for that attribute.

- Step 1: Define two anchor strings for each attribute:Positive Anchor: "The [Attribute] was perfect, excellent, and highly satisfying."Negative Anchor: "The [Attribute] was terrible, disappointing, and very poor."
- Step 2: Generate embeddings for the review and both anchors.
- Step 3: Calculate the Cosine Similarity ($cos(\theta)$) between the review and both anchors.
- Step 4: Map the relative similarity to your 1–9 scale.$$Score = 1 + 8 \times \left( \frac{Sim_{pos}}{Sim_{pos} + Sim_{neg}} \right)$$Why this works: It doesn't require you to chop up the review. The model naturally weighs the parts of the text that relate to the attribute keywords.

In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", cache_folder='../embedding_models').to(device)

## Study 1

In [12]:
df_1 = pd.read_csv("../data/predicted_attributes/predicted_Study_1_reviews.csv")

In [13]:
reviews = df_1["finalReview"].fillna("").tolist() 
review_embeddings = model.encode(reviews, convert_to_tensor=True, show_progress_bar=True).cpu().numpy()

df_1["review_embedding"] = list(review_embeddings)

Batches: 100%|██████████| 82/82 [00:01<00:00, 62.22it/s]


In [14]:
# for each attribute, a very positive example and a very negative example
attribute_anchors = {
    "cleaning_service_quality": [
        "Horrible quality wash and folding, clothes came back dirty and not ironed.",
        "The cleaning service quality was perfect, spotless, and excellent."
    ],
    "order_packaging": [
        "The order packaging was damaged, messy, and poorly handled.",
        "The order packaging was secure, neat, and very professional."
    ],
    "communication_and_responsiveness": [
        "they never replied",
        "they replied instantly"
    ],
    "Driver_professionalism": [
        "not helpful",
        "your drivers are always super nice and very professional"
    ],
    "Service_speed": [
        "hasn't arrived, lot of delays",
        "on time, efficient and fast"
    ],
}

anchor_embeddings = {}
for attr, phrases in attribute_anchors.items():
    anchor_embeddings[attr] = model.encode(phrases, convert_to_tensor=True)

In [15]:
import numpy as np

def calculate_attr_score(row, attr_name):
    if row.get(attr_name, 0) == 0: 
        row[f"{attr_name}_sentiment"] = np.nan
        return row
    
    rev_embed = torch.tensor(row["review_embedding"]).to(device)
    neg_embed = anchor_embeddings[attr_name][0]
    pos_embed = anchor_embeddings[attr_name][1]
    
    # 1. Calculate similarities and clamp to 0
    sim_neg = max(0, util.cos_sim(rev_embed, neg_embed).item())
    sim_pos = max(0, util.cos_sim(rev_embed, pos_embed).item())

    # 2. Handle the "No Match" case (both are 0) to avoid division by zero
    if sim_pos == 0 and sim_neg == 0:
        relative_score = 0.5  # Neutral default
    else:
        relative_score = sim_pos / (sim_pos + sim_neg)

    # 3. Scale to 1-9
    score = 1 + (8 * relative_score)
    row[f"{attr_name}_sentiment"] = round(score, 2)
    return row
    
# 6. Apply to the DataFrame
for attr in attribute_anchors.keys():
    # Use .apply with axis=1 for row-wise operations
    df_1 = df_1.apply(lambda row: calculate_attr_score(row, attr), axis=1)

In [16]:
df_1.columns

Index(['Unnamed: 0', 'ID', 'finalReview', 'Satisfaction_final',
       'cleaning_service_quality', 'order_packaging',
       'communication_and_responsiveness', 'Driver_professionalism',
       'Service_speed', 'cleaning_service_quality_sentiment',
       'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM', 'lang',
       'review_embedding'],
      dtype='object')

In [17]:
df_1 = df_1.drop(columns=['Unnamed: 0','review_embedding'])

In [18]:
df_1.to_excel('../data/pred_attributes_and_sentiment/study_1_results.xlsx', index=False)